In [ ]:
"""
CTQN-style Contextual Bandit Classifier for HUMS2023 (Normal vs Faulty)

Overview:
This script reproduces, in a contextual bandit framing, the CTQN-style agent described in
“A Convolutional-Transformer Reinforcement Learning Agent for Rotating Machinery Fault Diagnosis”
(CTQN). The original paper did not release code; this implementation follows the
architecture and training details as closely as the article allows.

Although we use a DQN-like stack (conv + transformer + classifier), the discount factor is 0
and each sample is labeled once with no temporal credit assignment. Practically, this reduces to
a single-step contextual bandit that chooses an action (label) per signal segment.

"""

import os, re, json, time, math, csv, random, glob
from pathlib import Path
import numpy as np
from scipy.io import loadmat

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from typing import List, Tuple, Optional

PATH_TRAIN_NORMAL = './data/Contextual_bandit/Normal_19_20' #normal
PATH_TRAIN_FAULTY = './data/Contextual_bandit/Faulty_26_27' #faulty
PATH_TEST_UNKNOWN = './data/Contextual_bandit/unknown21_25' #unknown


MAT_VAR_CANDIDATES = ["xr", "unsw", "ims", "xjtusy"]

OUT_DIR = "./ctqn_binary_outputs"


NORM_MODE = "off"

CONV_FILTERS_MODE = "small"
VAL_SPLIT = 0.1


SEED = 101
BATCH_SIZE = 32
NUM_EPOCHS = 300 #
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10  


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_NAMES = ["Normal", "Faulty"]  # 0,1

os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Out dir: {OUT_DIR}")



_nat_chunk = re.compile(r'(\d+)')
def natural_key(s: str):
    return [int(text) if text.isdigit() else text.lower()
            for text in _nat_chunk.split(s)]

def list_mat_files(folder: str) -> List[str]:
    paths = glob.glob(os.path.join(folder, "*.mat"))
  
    paths.sort(key=lambda p: natural_key(os.path.basename(p)))
    return paths

def load_mat_1d_signal(path: str, var_candidates: List[str]) -> np.ndarray:
   
    mat = loadmat(path)
    for var in var_candidates:
        if var in mat:
            arr = mat[var]
            
            arr = np.asarray(arr).squeeze()
            arr = arr.astype(np.float32)
            if arr.ndim != 1:
                raise ValueError(f"{path}: variable '{var}' is not 1-D after squeeze (shape={arr.shape})")
            return arr
   
    for k, v in mat.items():
        if k.startswith("__"):
            continue
        arr = np.asarray(v).squeeze()
        if arr.ndim == 1 and arr.size > 0 and np.issubdtype(arr.dtype, np.number):
            return arr.astype(np.float32)
    raise KeyError(f"No suitable 1-D vector found in {path}. Tried {var_candidates}")

def reshape_to_img(sig1d: np.ndarray) -> np.ndarray:
    """
    Row-major reshape:
      - 4096 -> (64, 64) # IMS and XJTUSY
      - 4095 -> (63, 65) # HUMS2023
    """
    n = sig1d.size
    if n == 4096:
        img = sig1d.reshape(64, 64)
    elif n == 4095:
        img = sig1d.reshape(63, 65)
    else:
        raise ValueError(f"Unsupported length {n}. Expected 4096 or 4095.")
    return img.astype(np.float32)


def compute_train_mu_sigma(train_signals_1d: List[np.ndarray]) -> Tuple[float, float]:
   
    all_concat = np.concatenate(train_signals_1d, axis=0).astype(np.float64)
    mu = float(all_concat.mean())
    sigma = float(all_concat.std() + 1e-8)
    return mu, sigma

def zscore_per_sample(x: np.ndarray) -> np.ndarray:
    mu = float(x.mean())
    sigma = float(x.std() + 1e-8)
    return (x - mu) / sigma

def zscore_with_mu_sigma(x: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    return (x - mu) / (sigma + 1e-8)



train_normal_files = list_mat_files(PATH_TRAIN_NORMAL)
train_faulty_files = list_mat_files(PATH_TRAIN_FAULTY)
test_unknown_files = list_mat_files(PATH_TEST_UNKNOWN)

print(f"Found train_normal: {len(train_normal_files)}, train_faulty: {len(train_faulty_files)}, test_unknown: {len(test_unknown_files)}")


train_normal_1d = [load_mat_1d_signal(p, MAT_VAR_CANDIDATES) for p in train_normal_files]
train_faulty_1d = [load_mat_1d_signal(p, MAT_VAR_CANDIDATES) for p in train_faulty_files]


scaler = {"mode": NORM_MODE, "mu": None, "sigma": None}
if NORM_MODE == "train_only":
    mu, sigma = compute_train_mu_sigma(train_normal_1d + train_faulty_1d)
    scaler["mu"], scaler["sigma"] = mu, sigma
    print(f"[Scaler] train_only mu={mu:.6f}, sigma={sigma:.6f}")
else:
    print(f"[Scaler] mode={NORM_MODE}")


def prep_sample(sig1d: np.ndarray) -> np.ndarray:
    x = sig1d.copy()
    if NORM_MODE == "per_sample":
        x = zscore_per_sample(x)
    elif NORM_MODE == "train_only":
        x = zscore_with_mu_sigma(x, scaler["mu"], scaler["sigma"])
   
    x2d = reshape_to_img(x)
    
    return np.expand_dims(x2d, axis=0)


X_train = []
y_train = []
train_filenames = []

for p in train_normal_files:
    sig = load_mat_1d_signal(p, MAT_VAR_CANDIDATES)
    X_train.append(prep_sample(sig))
    y_train.append(0)
    train_filenames.append(os.path.basename(p))

for p in train_faulty_files:
    sig = load_mat_1d_signal(p, MAT_VAR_CANDIDATES)
    X_train.append(prep_sample(sig))
    y_train.append(1)
    train_filenames.append(os.path.basename(p))

X_train = np.stack(X_train).astype(np.float32)  # (N,1,64,64)
y_train = np.array(y_train, dtype=np.int64)


X_unk = []
unk_filenames = []
for p in test_unknown_files:
    sig = load_mat_1d_signal(p, MAT_VAR_CANDIDATES)
    X_unk.append(prep_sample(sig))
    unk_filenames.append(os.path.basename(p))
X_unk = np.stack(X_unk).astype(np.float32) if len(X_unk) > 0 else np.zeros((0,1,64,64), dtype=np.float32)


X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)

class NumpyTensorDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor, names: List[str]):
        self.X = X
        self.y = y
        self.names = names
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.names[idx]

full_ds = NumpyTensorDataset(X_train_t, y_train_t, train_filenames)


if VAL_SPLIT > 0:
    n_total = len(full_ds)
    n_val = int(round(n_total * VAL_SPLIT))
    n_train = n_total - n_val
    train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
else:
    train_ds, val_ds = full_ds, None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True) if val_ds is not None else None


X_unk_t = torch.from_numpy(X_unk)
class UnknownDataset(Dataset):
    def __init__(self, X: torch.Tensor, names: List[str]):
        self.X = X
        self.names = names
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.names[idx]

unk_ds = UnknownDataset(X_unk_t, unk_filenames)
unk_loader = DataLoader(unk_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


cfg = {
    "paths": {
        "train_normal": PATH_TRAIN_NORMAL,
        "train_faulty": PATH_TRAIN_FAULTY,
        "test_unknown": PATH_TEST_UNKNOWN
    },
    "normalization": scaler,
    "conv_filters_mode": CONV_FILTERS_MODE,
    "val_split": VAL_SPLIT,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
}
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(cfg, f, indent=2)
print("Datasets ready.")


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  
        pe[:, 1::2] = torch.cos(position * div_term)  
        self.register_buffer("pe", pe.unsqueeze(0))   

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        L = x.size(1)
        return x + self.pe[:, :L, :]

class CTQNBinary(nn.Module):
    
    def __init__(self, conv_filters=(16,32), d_model=16, n_heads=4, ff_hidden=64, n_layers=2, num_classes=2):
        super().__init__()
        c1, c2 = conv_filters
        self.conv1 = nn.Conv2d(1, c1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  
        self.fix_tokens = nn.AdaptiveAvgPool2d((16, 16)) 
        self.seq_len = 16 * 16

        
        self.seq_len = 16 * 16
        self.proj = nn.Linear(c2, d_model)
        self.pos = PositionalEncoding(d_model=d_model, max_len=self.seq_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                                   dim_feedforward=ff_hidden, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc_out = nn.Linear(d_model, num_classes)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,1,64,64)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)  
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = self.fix_tokens(x)  
        
        B, C, H, W = x.shape

        x = x.view(B, C, H*W).permute(0, 2, 1) 
        x = self.proj(x)                        
        x = self.pos(x)                         
        x = self.transformer(x)                
        x = x.mean(dim=1)                       
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.forward_features(x)
        logits = self.fc_out(feats)             
        return logits

# Instantiate
conv_filters = (16,32) if CONV_FILTERS_MODE == "small" else (32,64)
model = CTQNBinary(conv_filters=conv_filters, d_model=16, n_heads=4, ff_hidden=64, n_layers=2, num_classes=2).to(DEVICE)
print(model)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val = float("inf")
best_epoch = -1
best_path = os.path.join(OUT_DIR, "ctqn_binary_best.pt")
history = {"train_loss": [], "val_loss": []}

def run_one_epoch(loader, train: bool):
    if train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for xb, yb, _names in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += float(loss.item()) * xb.size(0)
        preds = logits.argmax(dim=1)
        total_correct += int((preds == yb).sum().item())
        total += xb.size(0)
    avg_loss = total_loss / max(1, total)
    acc = total_correct / max(1, total)
    return avg_loss, acc

start = time.time()
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_one_epoch(train_loader, train=True)
    if val_loader is not None:
        va_loss, va_acc = run_one_epoch(val_loader, train=False)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        improved = va_loss < best_val
        metric_str = f"val_loss={va_loss:.4f}, val_acc={va_acc:.4f}"
    else:
      
        history["train_loss"].append(tr_loss)
        improved = tr_loss < best_val
        metric_str = f"train_loss={tr_loss:.4f}, train_acc={tr_acc:.4f}"

    if improved:
        best_val = va_loss if val_loader is not None else tr_loss
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save({
            "model_state": model.state_dict(),
            "conv_filters": conv_filters,
            "d_model": 16,
            "n_heads": 4,
            "ff_hidden": 64,
            "n_layers": 2,
            "num_classes": 2,
            "label_names": LABEL_NAMES,
            "scaler": scaler,
        }, best_path)
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:03d} | train_loss={tr_loss:.4f}, train_acc={tr_acc:.4f} | {metric_str} | best_epoch={best_epoch:03d}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping (no improvement for {PATIENCE} epochs).")
        break

elapsed = time.time() - start
print(f"Training done in {elapsed/60:.1f} min. Best epoch: {best_epoch}, checkpoint: {best_path}")

# Save training history
with open(os.path.join(OUT_DIR, "train_history.json"), "w") as f:
    json.dump(history, f, indent=2)


In [ ]:
# Load best model
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

pred_rows = []
total_normal = 0
total_faulty = 0

softmax = nn.Softmax(dim=1)

order_index = 0
with torch.no_grad():
    for xb, names in unk_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        probs = softmax(logits)  # (B,2)
        preds = probs.argmax(dim=1).cpu().numpy()
        probs_np = probs.cpu().numpy()

        for i in range(xb.size(0)):
            pred_idx = int(preds[i])
            pred_label = LABEL_NAMES[pred_idx]
            score = float(probs_np[i, pred_idx])
            fname = names[i]

            pred_rows.append({
                "order_index": order_index,
                "filename": fname,
                "pred_label": pred_label,
                "score": f"{score:.6f}",
            })
            order_index += 1

            if pred_idx == 0:
                total_normal += 1
            else:
                total_faulty += 1

# Save CSV
csv_path = os.path.join(OUT_DIR, "predictions_unknown.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_index", "filename", "pred_label", "score"])
    writer.writeheader()
    for row in pred_rows:
        writer.writerow(row)

# Save counts
counts = {"Normal": total_normal, "Faulty": total_faulty, "Total": total_normal + total_faulty}
with open(os.path.join(OUT_DIR, "summary_counts.json"), "w") as f:
    json.dump(counts, f, indent=2)

print(f"Saved predictions: {csv_path}")
print(f"Counts -> Normal: {total_normal} | Faulty: {total_faulty} | Total: {total_normal + total_faulty}")
